# Assignment 8 — Neural Machine Translation with OpenNMT-py (English → Hindi)
**ILLM** · **U23AI032**

* **Data:** `ilci` subset of BPCC (`ai4bharat/BPCC`), English source → Hindi target
  (change `TGT_LANG` to use another native language, e.g. `mar_Deva`, `ben_Beng`).
* **Split:** length-binned train / test (4 bins on source length), so each split has sentences of
  every length range.
* **Model:** OpenNMT-py attention-based encoder–decoder — 2-layer bidirectional LSTM encoder,
  2-layer LSTM decoder, Luong (general) attention with input feeding, on SentencePiece subwords.
* **Evaluation:** `sacrebleu` corpus BLEU (plus chrF), reported **per length bin**.

Run on a GPU runtime (e.g. Colab T4). OpenNMT-py 3.5.1 pins an older `torch`; if the runtime
complains after the install, restart it once and re-run from the top.

In [ ]:
!pip install -q "OpenNMT-py==3.5.1" "numpy<2" sacrebleu sentencepiece huggingface_hub pandas matplotlib

## 0. Settings and imports

In [ ]:
import os, re, csv, glob

import numpy as np
import pandas as pd
import torch
import yaml
import sentencepiece as spm

SEED = 42
TGT_LANG = "hin_Deva"            # native language = target, English = source

HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "mt_data")      # shared with Lab 9
RUN_DIR = os.path.join(HERE, "lab8_rnn")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

BINS = ["bin1", "bin2", "bin3", "bin4"]
TEST_PER_BIN = 500
VALID_SIZE = 1000

SPM_VOCAB = 8000
TRAIN_STEPS = 20000

USE_GPU = torch.cuda.is_available()
GPU_FLAG = "-gpu 0" if USE_GPU else ""
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if USE_GPU else "none")

## 1. Load the BPCC **ilci** subset (English → native language)

BPCC is a gated dataset: accept its terms at https://huggingface.co/datasets/ai4bharat/BPCC and
log in (`huggingface-cli login`, or set `HF_TOKEN`) before running this cell.

In [ ]:
from huggingface_hub import hf_hub_download

tsv_path = hf_hub_download("ai4bharat/BPCC", f"ilci/{TGT_LANG}.tsv", repo_type="dataset")
raw = pd.read_csv(tsv_path, sep="\t", quoting=csv.QUOTE_NONE, dtype=str,
                  keep_default_na=False, on_bad_lines="skip")
print("columns:", raw.columns.tolist(), "| rows:", len(raw))
raw.head()

In [ ]:
def latin_ratio(texts):
    s = "".join(texts)
    return sum(c.isascii() for c in s) / max(len(s), 1)


# the English column is the one written (mostly) in ASCII
cols = ["src", "tgt"] if {"src", "tgt"} <= set(raw.columns) else list(raw.columns[:2])
en_col = max(cols, key=lambda c: latin_ratio(raw[c].head(500)))
tgt_col = [c for c in cols if c != en_col][0]


def clean(s):
    return re.sub(r"\s+", " ", s).strip()


data = pd.DataFrame({"en": raw[en_col].map(clean), "tgt": raw[tgt_col].map(clean)})
data = data[(data.en != "") & (data.tgt != "")].drop_duplicates("en").reset_index(drop=True)
print(f"English column = {en_col!r}, {TGT_LANG} column = {tgt_col!r}")
print("clean sentence pairs:", len(data))
data.sample(5, random_state=SEED)

## 2. Length bins and the train / valid / test split

Sentence length = number of whitespace tokens in the **English (source)** sentence.
Lengths are integers, so the bins are read as:

| bin | length |
|-----|--------|
| bin1 | ≤ 10 |
| bin2 | 11 – 20 |
| bin3 | 21 – 29 |
| bin4 | ≥ 30 |

The split is **not** a plain random split: every bin is split separately, so both train and test
contain sentences from all four length ranges. The test set takes the same number of sentences
from each bin (`TEST_PER_BIN`, capped at 10 % of a bin), so every bin's BLEU is computed on a
comparably sized sample. A small validation set (for OpenNMT's periodic validation) is carved out
of the training part.

In [ ]:
def length_bin(n):
    if n <= 10:
        return "bin1"
    if n <= 20:
        return "bin2"
    if n < 30:
        return "bin3"
    return "bin4"


data["length"] = data.en.str.split().map(len)
data["bin"] = data.length.map(length_bin)

rng = np.random.default_rng(SEED)
train_parts, test_parts = [], []
for b in BINS:
    group = data[data.bin == b]
    group = group.iloc[rng.permutation(len(group))]
    n_test = min(TEST_PER_BIN, len(group) // 10)
    test_parts.append(group.iloc[:n_test])
    train_parts.append(group.iloc[n_test:])

train_all = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
valid_set = train_all.iloc[:VALID_SIZE].reset_index(drop=True)
train_set = train_all.iloc[VALID_SIZE:].reset_index(drop=True)
test_set = pd.concat(test_parts).reset_index(drop=True)          # ordered bin1 → bin4

summary = pd.DataFrame({name: split.bin.value_counts().reindex(BINS)
                        for name, split in [("train", train_set), ("valid", valid_set),
                                            ("test", test_set)]})
summary.loc["total"] = summary.sum()
summary

In [ ]:
def write_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")


def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]


for name, split in [("train", train_set), ("valid", valid_set), ("test", test_set)]:
    write_lines(os.path.join(DATA_DIR, f"{name}.en"), split.en)
    write_lines(os.path.join(DATA_DIR, f"{name}.tgt"), split.tgt)
write_lines(os.path.join(DATA_DIR, "test.bins"), test_set.bin)

test_bins = list(test_set.bin)
print(sorted(os.listdir(DATA_DIR)))

## 3. Subword segmentation (SentencePiece)

A separate unigram SentencePiece model is trained on each side of the **training** data. Working on
subwords keeps the vocabulary small and removes the unknown-word problem, which matters for a
morphologically rich target like Hindi. The test outputs are decoded back to plain text before
scoring, so BLEU is computed on normal detokenised sentences.

In [ ]:
sp = {}
for side in ["en", "tgt"]:
    prefix = os.path.join(RUN_DIR, f"spm.{side}")
    spm.SentencePieceTrainer.train(
        input=os.path.join(DATA_DIR, f"train.{side}"), model_prefix=prefix,
        vocab_size=SPM_VOCAB, model_type="unigram", character_coverage=1.0,
        hard_vocab_limit=False, input_sentence_size=1_000_000,
        shuffle_input_sentence=True, minloglevel=2)
    sp[side] = spm.SentencePieceProcessor(model_file=prefix + ".model")

for name in ["train", "valid", "test"]:
    for side in ["en", "tgt"]:
        lines = read_lines(os.path.join(DATA_DIR, f"{name}.{side}"))
        pieces = sp[side].encode(lines, out_type=str)
        write_lines(os.path.join(RUN_DIR, f"{name}.sp.{side}"), [" ".join(p) for p in pieces])

print(read_lines(os.path.join(RUN_DIR, "train.sp.en"))[0])
print(read_lines(os.path.join(RUN_DIR, "train.sp.tgt"))[0])

## 4. OpenNMT-py configuration

| component | setting |
|-----------|---------|
| encoder | bidirectional LSTM, 2 layers, 512 hidden |
| decoder | LSTM, 2 layers, 512 hidden, input feeding |
| attention | Luong *general* |
| embeddings | 256-d, learned from scratch |
| optimiser | Adam, lr 1e-3, batch 64 sentences, dropout 0.3 |
| training | 20k steps, validation every 1k steps |

In [ ]:
config = {
    "save_data": os.path.join(RUN_DIR, "data"),
    "src_vocab": os.path.join(RUN_DIR, "vocab.src"),
    "tgt_vocab": os.path.join(RUN_DIR, "vocab.tgt"),
    "overwrite": True,
    "data": {
        "corpus_1": {"path_src": os.path.join(RUN_DIR, "train.sp.en"),
                     "path_tgt": os.path.join(RUN_DIR, "train.sp.tgt"),
                     "transforms": ["filtertoolong"]},
        "valid": {"path_src": os.path.join(RUN_DIR, "valid.sp.en"),
                  "path_tgt": os.path.join(RUN_DIR, "valid.sp.tgt")},
    },
    "src_seq_length": 200,
    "tgt_seq_length": 200,

    # model
    "encoder_type": "brnn",
    "decoder_type": "rnn",
    "rnn_type": "LSTM",
    "enc_layers": 2,
    "dec_layers": 2,
    "hidden_size": 512,
    "word_vec_size": 256,
    "global_attention": "general",
    "input_feed": 1,
    "dropout": [0.3],

    # training
    "save_model": os.path.join(RUN_DIR, "model"),
    "train_steps": TRAIN_STEPS,
    "valid_steps": 1000,
    "save_checkpoint_steps": 5000,
    "keep_checkpoint": 2,
    "report_every": 500,
    "optim": "adam",
    "learning_rate": 0.001,
    "batch_type": "sents",
    "batch_size": 64,
    "valid_batch_size": 64,
    "max_grad_norm": 5,
    "seed": SEED,
}
if USE_GPU:
    config.update({"world_size": 1, "gpu_ranks": [0]})

CONFIG_PATH = os.path.join(RUN_DIR, "rnn.yaml")
with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)
print(open(CONFIG_PATH).read())

## 5. Build vocabulary and train

In [ ]:
!onmt_build_vocab -config {CONFIG_PATH} -n_sample -1

In [ ]:
!onmt_train -config {CONFIG_PATH}

## 6. Translate the test set

In [ ]:
def latest_checkpoint(run_dir):
    ckpts = glob.glob(os.path.join(run_dir, "model_step_*.pt"))
    return max(ckpts, key=lambda p: int(re.search(r"step_(\d+)", p).group(1)))


CKPT = latest_checkpoint(RUN_DIR)
TEST_SRC = os.path.join(RUN_DIR, "test.sp.en")
PRED_SP = os.path.join(RUN_DIR, "pred.sp.tgt")
print("checkpoint:", CKPT)

In [ ]:
!onmt_translate -model {CKPT} -src {TEST_SRC} -output {PRED_SP} -beam_size 5 -batch_size 64 -max_length 250 {GPU_FLAG}

In [ ]:
hyps = [sp["tgt"].decode_pieces(line.split()) for line in read_lines(PRED_SP)]
refs = read_lines(os.path.join(DATA_DIR, "test.tgt"))
srcs = read_lines(os.path.join(DATA_DIR, "test.en"))
write_lines(os.path.join(RUN_DIR, "pred.tgt"), hyps)
assert len(hyps) == len(refs) == len(test_bins)

## 7. Bin-wise BLEU (sacrebleu)

In [ ]:
import sacrebleu


def binwise_scores(hyps, refs, bins):
    rows = []
    for b in BINS + ["all"]:
        idx = [i for i, x in enumerate(bins) if b == "all" or x == b]
        h = [hyps[i] for i in idx]
        r = [refs[i] for i in idx]
        bleu = sacrebleu.corpus_bleu(h, [r])
        chrf = sacrebleu.corpus_chrf(h, [r])
        rows.append({"bin": b, "sentences": len(idx), "BLEU": round(bleu.score, 2),
                     "chrF": round(chrf.score, 2), "BP": round(bleu.bp, 3),
                     "hyp/ref len": round(bleu.sys_len / max(bleu.ref_len, 1), 3),
                     "1/2/3/4-gram prec": "/".join(f"{p:.1f}" for p in bleu.precisions)})
    return pd.DataFrame(rows).set_index("bin")

In [ ]:
scores = binwise_scores(hyps, refs, test_bins)
scores.to_csv(os.path.join(RUN_DIR, "binwise_bleu.csv"))
scores

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
labels = ["bin1\n≤10", "bin2\n11–20", "bin3\n21–29", "bin4\n≥30"]
ax.bar(labels, scores.loc[BINS, "BLEU"], color="#4C72B0")
ax.axhline(scores.loc["all", "BLEU"], color="grey", ls="--", label=f"overall {scores.loc['all', 'BLEU']}")
for i, v in enumerate(scores.loc[BINS, "BLEU"]):
    ax.text(i, v + 0.3, f"{v:.1f}", ha="center")
ax.set_ylabel("BLEU")
ax.set_title("LSTM + attention: BLEU by source length")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for b in BINS:
    i = test_bins.index(b)
    print(f"--- {b} ---")
    print("SRC :", srcs[i])
    print("REF :", refs[i])
    print("HYP :", hyps[i])

## 8. Bin-wise analysis

*(Read together with the table and plot above — the numbers come from the run.)*

**What the four bins show**

* **bin1 (≤ 10 words).** Short sentences are easy to translate *word-for-word*, but BLEU on them is
  fragile: each sentence has only a handful of 4-grams, so one wrong or reordered word removes a
  large share of the matches, and the brevity penalty (BP) is applied harshly if the model drops a
  word. BLEU here is therefore not always the highest even though the translations look good —
  chrF, which is character-based, is the steadier metric for this bin.
* **bin2 (11–20 words)** and **bin3 (21–29 words)** cover most of ILCI's tourism/health sentences, so
  the model sees the most training data at these lengths. They typically give the best (or close
  to best) BLEU of the four bins.
* **bin4 (≥ 30 words).** This is where the RNN encoder–decoder is weakest. The encoder has to carry
  the whole sentence through a recurrent state, and although attention helps, long sentences show
  the familiar failure modes: repeated phrases, dropped clauses (watch for a falling `hyp/ref len`
  ratio and BP < 1), and wrong long-distance agreement. English is SVO while Hindi is SOV, so long
  sentences also need more reordering. These sentences are also the rarest in training.

**Overall pattern.** Expect BLEU to rise from bin1 to the middle bins and then fall for bin4. The
fall on bin4 is the main result: an attention LSTM generalises poorly to long inputs, and the
Transformer in Lab 9 is meant to fix this. The n-gram precision column shows the same effect:
4-gram precision falls fastest with length, because long-range word order is what the model
gets wrong.